In [1]:
import pandas as pd
from utils.utils import setup_model, pull_spectral_images, save_model
from sklearn.model_selection import train_test_split
from net.generator import SpectralImageDataGenerator
from net.net_def import AutoencoderHyperModel
from kerastuner.tuners import Hyperband, RandomSearch
import json


# Load the TensorBoard notebook extension
%load_ext tensorboard
import tensorflow as tf

In [2]:
# Model Configuration
MODEL_NAME = 'hyper_3d_autoencoder'
INPUT_FILES = "hyspecnet-11k/patches_tif_npy"
INPUT_FILE_TYPE = 'tif'
BATCH_SIZE = 32
EPOCHS = 10
USE_CALLBACKS = ['tf_logs', 'early_stopping']

In [3]:
# Load the spectral images
spectral_images, input_shape = pull_spectral_images(INPUT_FILES, type=INPUT_FILE_TYPE)
print(f"Loaded {len(spectral_images)} spectral images with input shape {input_shape}.")

Pulling spectral images from .TIF files
Loaded 187 spectral images with input shape (128, 128, 224, 1).


In [4]:
# Split the data into training and validation sets
image_paths_train, image_paths_val = train_test_split(spectral_images, test_size=0.2, random_state=42)
train_gen = SpectralImageDataGenerator(image_paths_train, batch_size=BATCH_SIZE)
val_gen = SpectralImageDataGenerator(image_paths_val, batch_size=BATCH_SIZE)
print(f"Inputs set to {input_shape} with {len(image_paths_train)} training images and {len(image_paths_val)} validation images.")

Creating generator with 149 images. Batch size: 32 Shuffle: False
Creating generator with 38 images. Batch size: 32 Shuffle: False
Inputs set to (128, 128, 224, 1) with 149 training images and 38 validation images.


In [5]:
model_dir, callbacks = setup_model(MODEL_NAME, use_callbacks=USE_CALLBACKS)

hypermodel = AutoencoderHyperModel(input_shape=input_shape)  # Adjusted to use the dynamic input_shape

tuner = Hyperband(
    hypermodel,
    objective='val_mae',
    directory=model_dir,
    project_name='tuning',
    max_consecutive_failed_trials=50
)

%tensorboard --logdir {model_dir} 
# Start the search process
tuner.search(train_gen, validation_data=val_gen, epochs=EPOCHS, callbacks=callbacks)

Trial 22 Complete [00h 00m 07s]

Best val_mae So Far: 0.21710734069347382
Total elapsed time: 07h 58m 51s


In [15]:
# After search, retrieve the best model
best_model = tuner.get_best_models(num_models=1)[0]
best_hyperparameters = tuner.get_best_hyperparameters()[0]
best_hyperparameters.values

/Users/jw/.pyenv/versions/bnjo_net/lib/python3.11/site-packages/keras/src/saving/saving_lib.py:396: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 38 variables. 
  trackable.load_own_variables(weights_store.get(inner_path))


{'conv_layers': 3,
 'filters_0': 64,
 'dense_units': 192,
 'convT_layers': 3,
 'filtersT_0': 16,
 'filters_1': 32,
 'filters_2': 48,
 'filtersT_1': 16,
 'filtersT_2': 16,
 'tuner/epochs': 10,
 'tuner/initial_epoch': 0,
 'tuner/bracket': 0,
 'tuner/round': 0}

In [17]:
# After tuning, save the best hyperparameters
saved_model = save_model(best_model, MODEL_NAME, model_dir)  
best_hyperparameters = tuner.get_best_hyperparameters()[0].get_config()
hp_filename = f"{model_dir}/best_hyperparameters.json"

with open(hp_filename, 'w') as f:
    json.dump(best_hyperparameters, f)

print(f"Best hyperparameters saved to {hp_filename}")
print(json.dumps(best_hyperparameters, indent=4))

Model saved to models/hyper_3d_autoencoder_20240314_180234/hyper_3d_autoencoder.keras
Best hyperparameters saved to models/hyper_3d_autoencoder_20240314_180234/best_hyperparameters.json
{
    "space": [
        {
            "class_name": "Int",
            "config": {
                "name": "conv_layers",
                "default": null,
                "conditions": [],
                "min_value": 1,
                "max_value": 3,
                "step": 1,
                "sampling": "linear"
            }
        },
        {
            "class_name": "Int",
            "config": {
                "name": "filters_0",
                "default": null,
                "conditions": [],
                "min_value": 16,
                "max_value": 64,
                "step": 16,
                "sampling": "linear"
            }
        },
        {
            "class_name": "Int",
            "config": {
                "name": "dense_units",
                "default": null,
     